In [ ]:
# ==========================================
# Google Colab 專用：OCR 後端 API (ngrok 永久隧道版)
# 包含 ROI 放大裁切技術，大幅提升左上角關卡名稱辨識率
# 內建 Numpy 衝突防禦與【自動重啟機制】
# ==========================================
import os
import sys
import subprocess
import time

# --- 智慧自動安裝與環境防禦區塊 ---
def check_and_install_environment():
    need_install = False
    try:
        import pyngrok
        import paddleocr
        import fastapi
        import cv2
        import numpy as np
        # 嚴格檢查 Numpy 是否為會導致當機的 2.x 版本
        if int(np.__version__.split('.')[0]) >= 2:
            need_install = True
    except ImportError:
        need_install = True

    if need_install:
        print("⏳ 發現全新機器！正在為您安裝專屬 AI 環境 (約需 1 分鐘)...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "numpy<2.0.0", "opencv-python-headless==4.8.0.74",
            "fastapi", "uvicorn==0.29.0", "python-multipart",
            "paddlepaddle==2.6.2", "paddleocr==2.7.3",
            "nest-asyncio", "pyngrok==7.1.6"
        ])
        print("\n" + "✨"*25)
        print("🚨 【系統自動重啟中，請稍候...】 🚨")
        print("套件安裝完畢！為了讓乾淨的環境生效，程式即將自動重啟核心...")
        print("👉 【重啟完成後，請再次點擊左側的播放鍵，即可順利啟動伺服器！】")
        print("✨"*25 + "\n")
        time.sleep(3)
        # 魔法機制：自動終止當前程序，觸發 Colab 自動重啟核心，省去手動點選單的麻煩！
        os.kill(os.getpid(), 9)

# 執行環境檢查 (如果環境完美，這步會瞬間閃過)
check_and_install_environment()


# ==========================================
# 正式主程式區 (必須在環境完美、重啟後才會執行到這裡)
# ==========================================
import nest_asyncio

# 必須在載入其他套件之前，先為 Colab 的非同步環境打上補丁
nest_asyncio.apply()

from fastapi import FastAPI, File, UploadFile, Form
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
from paddleocr import PaddleOCR
import numpy as np
import cv2
from pyngrok import ngrok

# ==========================================
# ⚙️ 個人化設定區
# ==========================================
# 🔐 從 Colab Secrets 讀取 ngrok 金鑰（不寫死在程式碼裡）
# 設定方式：左側工具列「🔑 Secrets」→ 新增名稱 NGROK_TOKEN、貼上金鑰 → 開啟本 notebook 的存取開關
from google.colab import userdata
try:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
except Exception:
    NGROK_TOKEN = None
if not NGROK_TOKEN:
    raise RuntimeError("❌ 找不到 NGROK_TOKEN！請到左側『🔑 Secrets』新增名為 NGROK_TOKEN 的密鑰，並開啟本 notebook 的存取權限後再執行。")
FIXED_DOMAIN = "flanking-snort-cyclic.ngrok-free.dev"

# 班級密碼（擋掉外部亂打 API）；同樣存在 Colab Secrets，名稱 CLASS_PASSCODE。留空則不啟用檢查。
try:
    CLASS_PASSCODE = (userdata.get('CLASS_PASSCODE') or '').strip()
except Exception:
    CLASS_PASSCODE = None

# 初始化 FastAPI
app = FastAPI(title="OCR API Server (ngrok版)")

# 允許跨來源請求 (CORS)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 載入模型 (防呆機制)
if 'ocr_model' not in globals():
    print("🚀 正在載入 PaddleOCR 模型，請稍候...")
    ocr_model = PaddleOCR(use_angle_cls=True, lang='chinese_cht')
    print("✅ 模型載入完成！")

# ==========================================
# 網頁路由 (API Endpoints)
# ==========================================
@app.get("/health")
def health_check():
    """提供前端檢查伺服器是否活著的端點"""
    return {"status": "ok", "message": "OCR API is running!"}

@app.post("/analyze")
async def analyze_image(
    file: UploadFile = File(...),
    keywords: str = Form(""),
    password: str = Form("")
):
    """處理前端傳來的圖片並進行辨識與判定"""
    # 班級密碼驗證（擋掉外部亂打 API）
    if CLASS_PASSCODE and (password or "").strip() != CLASS_PASSCODE:
        print("[班級密碼] 不符：請確認前端 CLASS_PASS 與 Secrets 的 CLASS_PASSCODE 完全一致"
              "（或刪除該 Secret 以停用檢查）")
        return {"status": "error", "message": "班級密碼錯誤，無法使用辨識服務。"}
    try:
        # 1. 讀取與轉換圖片
        contents = await file.read()
        nparr = np.frombuffer(contents, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        if img is None:
            return {"status": "error", "message": "無法解析圖片，請確保上傳的是有效圖檔。"}

        if len(img.shape) == 3 and img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)

        # === 核心升級：針對左上角裁切與放大 (ROI 強化) ===
        h, w = img.shape[:2]
        # 裁切左上角 (高度前 20%，寬度前 70%)
        top_left_roi = img[0:int(h*0.20), 0:int(w*0.70)]
        # 放大 2.5 倍以提升小字辨識率 (使用 INTER_CUBIC 演算法平滑補點)
        zoomed_roi = cv2.resize(top_left_roi, None, fx=2.5, fy=2.5, interpolation=cv2.INTER_CUBIC)

        # 2. 執行 OCR
        # 分別辨識「放大後的左上角」與「整張原圖」
        result_zoomed = ocr_model.ocr(zoomed_roi)
        result_full = ocr_model.ocr(img)

        extracted_texts = []

        # 優先把放大後得到的高精準度文字，放進清單的最前面
        if result_zoomed and result_zoomed[0]:
            for line in result_zoomed[0]:
                extracted_texts.append(line[1][0])

        # 再放入整張圖的文字 (用來確保能找到 "挑戰成功" 等位於中央的資訊)
        if result_full and result_full[0]:
            for line in result_full[0]:
                extracted_texts.append(line[1][0])

        # === 後端判定（改由後端裁決，前端無法竄改）===
        combined = "".join(extracted_texts).replace(" ", "").replace("\n", "")
        combined_lower = combined.lower()
        kw = [k.strip() for k in keywords.split(",") if k.strip()]
        success_kw = kw[0] if len(kw) > 0 else "挑戰成功"
        title = kw[1].replace(" ", "") if len(kw) > 1 else ""
        eng = kw[2].lower() if len(kw) > 2 else ""

        has_success = success_kw in combined
        has_level = bool(title) and (title in combined)
        if (not has_level) and title and eng:
            match = sum(1 for ch in title if ch in combined)
            rate = match / len(title) if title else 0
            last_two = eng.split()[-2:]
            has_eng = all(w in combined_lower for w in last_two) if last_two else False
            if rate >= 0.6 and has_eng:
                has_level = True

        passed = bool(has_success and has_level)
        reasons = []
        if not has_level:
            reasons.append("關卡名稱不符合")
        if not has_success:
            reasons.append("沒有偵測到『挑戰成功』")

        # 回傳判定結果 + 原始文字（raw_text 供老師稽核，判定以 pass 為準）
        return {
            "status": "success",
            "pass": passed,
            "reasons": reasons,
            "raw_text": extracted_texts
        }

    except Exception as e:
        import traceback
        return {"status": "error", "message": str(e), "trace": traceback.format_exc()}

# ==========================================
# 啟動伺服器與永久隧道
# ==========================================
if __name__ == "__main__":
    # 關閉之前可能遺留的 ngrok 連線
    os.system("killall ngrok")

    # 安全的關閉舊 ngrok 實例
    try:
        ngrok.kill()
    except:
        pass

    # 設定 ngrok auth token
    ngrok.set_auth_token(NGROK_TOKEN)

    try:
        # 建立固定網域的隧道
        public_url = ngrok.connect(8000, domain=FIXED_DOMAIN).public_url
        print("\n" + "="*60)
        print("🎉 【後端 API 伺服器啟動成功】 🎉")
        print(f"✅ 後端已永久發佈至: {public_url}")
        print("👉 您的前端網頁從今以後都不需要再更改網址了！")
        print("="*60 + "\n")

        # 啟動 API 伺服器
        uvicorn.run(app, host="0.0.0.0", port=8000)
    except Exception as e:
        print(f"❌ 隧道啟動失敗：{e}")
        print("請確認您的 NGROK_TOKEN 和 FIXED_DOMAIN 是否正確申請且可用。")